In [3]:
import pandas as pd
import numpy as np

print("--- Step 1:Data Loading & Initial Inspection ---")

df_raw = pd.read_csv('Sales Data Cleaning Task.csv')

print("--- Initial Data Inspection ---")
print(f"Shape: {df_raw.shape}")
print(df_raw.info())
print(df_raw.head())
df = df_raw.copy()
if len(df) < 120 or df.isna().sum().sum() == 0:
    print("WARNING: This file appears to be already cleaned.")
    print("Please restore the original raw 'Sales Data Cleaning Task.csv' to see the cleaning steps in action.")
    print("-" * 60)
else:
    print(f"Raw data loaded successfully: {len(df)} rows detected. Starting cleaning process...\n")


print("--- Step 2: Missing Values & Blank Data ---")
df = df.replace(r'^\s*$', np.nan, regex=True)
print("Missing values per column before handling:\n" + df.isna().sum().to_string())
for col in ['Region', 'Product', 'Salesperson']:
    df[col] = df[col].fillna('Unknown')
print("Total missing values after categorical imputation:", df.isna().sum().sum())


print("\n--- Step 3: Duplicate Records & Order IDs ---")
exact_dupes = df.duplicated().sum()
df = df.drop_duplicates(keep='first')
order_dupes = df.duplicated(subset=['Order_ID']).sum()
df = df.sort_values(by=['Order_ID', 'Sales'], na_position='last')
df = df.drop_duplicates(subset=['Order_ID'], keep='first')
print(f"Exact row duplicates removed: {exact_dupes}")
print(f"Duplicate Order IDs resolved: {order_dupes}")


print("\n--- Step 4: Data Type & Format Correction ---")
df['Order_Date'] = pd.to_datetime(df['Order_Date'], errors='coerce')
if df['Unit_Price'].dtype == 'O':
    df['Unit_Price'] = df['Unit_Price'].str.replace(r'[^\d.]', '', regex=True)
df['Unit_Price'] = pd.to_numeric(df['Unit_Price'], errors='coerce')
df['Quantity'] = pd.to_numeric(df['Quantity'], errors='coerce')
df['Discount'] = pd.to_numeric(df['Discount'], errors='coerce')
df['Sales'] = pd.to_numeric(df['Sales'], errors='coerce')
df['Profit'] = pd.to_numeric(df['Profit'], errors='coerce')
print("Data Types after conversion:\n" + df[['Order_Date', 'Unit_Price', 'Quantity', 'Sales']].dtypes.to_string())


print("\n--- Step 5: Text Standardization ---")
text_cols = ['Region', 'Product', 'Category', 'Salesperson']
print("Sample before standardization (Raw Category):", df['Category'].head(3).tolist())
for col in text_cols:
    df[col] = df[col].astype(str).str.strip().str.title()
print("Sample after standardization (Cleaned Category):", df['Category'].head(3).tolist())


print("\n--- Step 6: Invalid Value Detection ---")
neg_qty = (df['Quantity'] < 0).sum()
df['Quantity'] = df['Quantity'].abs()
df['Unit_Price'] = df['Unit_Price'].abs()
df['Discount'] = df['Discount'].abs()
high_disc = (df['Discount'] > 1).sum()
df.loc[df['Discount'] > 1, 'Discount'] = df['Discount'] / 100
rows_before_drop = len(df)
df = df.dropna(subset=['Order_Date', 'Quantity', 'Unit_Price'])
rows_after_drop = len(df)
print(f"Negative quantities corrected: {neg_qty}")
print(f"Discounts > 100% scaled to percentages: {high_disc}")
print(f"Rows dropped due to unrecoverable critical info (Date/Qty/Price): {rows_before_drop - rows_after_drop}")


print("\n--- Step 7: Business Rule Validation ---")
invalid_profit = (df['Profit'] > df['Sales']).sum()
expected_sales = df['Quantity'] * df['Unit_Price'] * (1 - df['Discount'])
df['Sales'] = expected_sales
df.loc[df['Profit'] > df['Sales'], 'Profit'] = np.nan
df['Profit'] = df['Profit'].fillna(df['Sales'] * 0.10)
print(f"Records where Profit exceeded Sales (capped/imputed): {invalid_profit}")

print("\n--- Steps 8 & 9: Final Data Quality Validation ---")
print("Final Missing Values:\n" + df.isna().sum().to_string())
print("\nDuplicate Order IDs:", df.duplicated(subset=['Order_ID']).sum())
print("Final Dataset Shape:", df.shape)
print("\nData Types Validated:\n" + df.dtypes.to_string())
print("\nCategory Consistency (Unique Regions):", df['Region'].unique().tolist())


print("\n--- Steps 10: Export & Document the Result ---")

df.to_csv('Sales Data Cleaned.csv', index=False)
print("\nCleaned dataset exported successfully to 'Sales Data Cleaned.csv'")

--- Step 1:Data Loading & Initial Inspection ---
--- Initial Data Inspection ---
Shape: (120, 11)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 120 entries, 0 to 119
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Order_ID     120 non-null    object 
 1   Order_Date   119 non-null    object 
 2   Region       118 non-null    object 
 3   Product      119 non-null    object 
 4   Category     120 non-null    object 
 5   Quantity     119 non-null    object 
 6   Unit_Price   119 non-null    object 
 7   Discount     120 non-null    float64
 8   Sales        119 non-null    float64
 9   Profit       119 non-null    float64
 10  Salesperson  118 non-null    object 
dtypes: float64(3), object(8)
memory usage: 10.4+ KB
None
  Order_ID  Order_Date Region     Product     Category Quantity Unit_Price  \
0    S0001  2026-01-27  North  Smartphone  Electronics      1.0    28000.0   
1    S0002  2026-06-04  North      La